In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Genera attendance_docs_enriched.json con:
  • fecha_1, fecha_2, fecha_larga, fecha_corta, hora
  • fecha_utc5 (sin modificaciones)              ← se conserva
Descarta registros con campos vacíos o nulos.
"""

import json
from pathlib import Path
import re
from datetime import datetime
import uuid

# ──────── Rutas ───────────────────────────────────────────────────────────────
json_dir = Path(
    "/home/nahumfg/Projects/GithubProjects/tesismaestriauni-launcher/"
    "publicdata-yolo-ocr/notebooks_ocr/asistencia/jsons/encabezado"
)
output_path = Path("../data/attendance_docs_enriched.json")

# ──────── Regex con grupos de captura ─────────────────────────────────────────
re_pp   = re.compile(r"pp(?P<start>\d{4})_(?P<end>\d{4})")
re_pa   = re.compile(r"pa(?P<start>\d{4})_(?P<end>\d{4})")
re_leg  = re.compile(r"leg(\d)")
re_page = re.compile(r"page_(\d+)")

MESES_ES    = ["enero","febrero","marzo","abril","mayo","junio",
               "julio","agosto","septiembre","octubre","noviembre","diciembre"]
MESES_ABBR  = ["ene","feb","mar","abr","may","jun",
               "jul","ago","sep","oct","nov","dic"]

records, omitidos = [], 0

for json_path in json_dir.glob("*.json"):
    try:
        with json_path.open(encoding="utf-8") as f:
            data = json.load(f)

        sesion      = json_path.stem
        fecha_iso   = data.get("fecha") or data.get("Fecha") or ""
        fecha_utc5  = data.get("utc_5") or data.get("utc5") or fecha_iso

        # ── Validación de la fecha ISO ───────────────────────────────────────
        try:
            dt = datetime.fromisoformat(fecha_iso)
        except Exception:
            omitidos += 1
            continue

        # ── Nuevos formatos ─────────────────────────────────────────────────
        fecha_1     = dt.strftime("%Y-%m-%d")
        fecha_2     = dt.strftime("%d/%m/%Y")
        fecha_larga = f"{dt.day:02d} de {MESES_ES[dt.month-1]} del {dt.year}"
        fecha_corta = f"{dt.day:02d} {MESES_ABBR[dt.month-1]} {dt.year}"
        hora        = dt.strftime("%H:%M")

        # ── Períodos y demás metadatos ──────────────────────────────────────
        pp_match = re_pp.search(sesion)
        pa_match = re_pa.search(sesion)

        record = {
            "id": str(uuid.uuid4()),
            "sesion": sesion,
            "fecha": fecha_iso,
            "fecha_utc5": fecha_utc5,
            "fecha_yyyymmdd": fecha_1,
            "fecha_ddmmyyyy": fecha_2,
            "fecha_larga": fecha_larga,
            "fecha_corta": fecha_corta,
            "hora": hora,
            "legislatura": data.get("legislatura", ""),
            "periodo_congreso_inicio": int(pp_match.group("start")) if pp_match else None,
            "periodo_congreso_fin": int(pp_match.group("end"))   if pp_match else None,
            "periodo_congreso": (f"{pp_match.group('start')}-{pp_match.group('end')}"
                                 if pp_match else ""),
            "periodo_anual_inicio": int(pa_match.group("start")) if pa_match else None,
            "periodo_anual_fin": int(pa_match.group("end"))     if pa_match else None,
            "periodo_anual": (f"{pa_match.group('start')}-{pa_match.group('end')}"
                              if pa_match else ""),
            "n_legislatura": int(re_leg.search(sesion).group(1)) if re_leg.search(sesion) else None,
            "page": int(re_page.search(sesion).group(1)) if re_page.search(sesion) else None,
            "url": f"http://localhost:8080/asistencia/{sesion}.png",
        }

        # Validar que no haya None ni "" en los valores
        if all(v not in (None, "") for v in record.values()):
            records.append(record)
        else:
            omitidos += 1

    except Exception as e:
        print(f"⚠️  Error en {json_path.name}: {e}")
        omitidos += 1

# ──────── Guardado ────────────────────────────────────────────────────────────
records.sort(key=lambda r: r["fecha_yyyymmdd"])
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"✅ Generado {output_path} ({len(records)} registros válidos)")
print(f"❌ Registros omitidos por campos vacíos o nulos: {omitidos}")


✅ Generado ../data/attendance_docs_enriched.json (847 registros válidos)
❌ Registros omitidos por campos vacíos o nulos: 12
